
%md
# Bronze — tabelas de referência

Quatro tabelas que dão **nome** ao que o VRA guarda como **código**:

| tabela | de onde vem | resolve |
|---|---|---|
| `voebem.bronze.aerodromos` | `AerodromosPublicos.csv` | `SBGR` → Guarulhos / São Paulo / SP |
| `voebem.bronze.empresas_nacionais` | `pda_empresas_aereas_nacionais.csv` | `GLO` → GOL LINHAS AÉREAS S.A. |
| `voebem.bronze.empresas_estrangeiras` | `pda_empresas_aereas_estrangeiros.csv` | `TAP` → TAP TRANSPORTES AÉREOS PORTUGUESES |
| `voebem.bronze.codigos_operacao` | seed table curada | `N` → Doméstica Mista |

 **Um arquivo, uma tabela.** Empresas aéreas chegam em DOIS cadastros da ANAC, e no
 bronze elas continuam em duas tabelas — bronze preserva o dado como chegou. Unir os
 dois é decisão de modelagem, e decisão de modelagem é trabalho da silver (marco-05).

 Cada arquivo tem uma armadilha diferente. Mesmo órgão, mesmo portal.


In [0]:
# Importa as funções do PySpark para manipulação de dados
from pyspark.sql import functions as F

# Define o caminho base onde estão os arquivos de referência no volume Unity Catalog
REF = "/Volumes/voebem/bronze/arquivos/referencias"

# Define um caractere que NÃO existe no arquivo (NUL) para desligar o quoting do leitor de CSV
# Necessário porque os arquivos usam aspas (") como símbolo de segundos nas coordenadas (09°52'06"S)
SEM_ASPAS = chr(0)


## 1. Aeródromos — latin-1 e aspas que não são aspas

Duas armadilhas neste arquivo:

**`encoding = "ISO-8859-1"`.** Este CSV não é UTF-8. Lido como UTF-8, "Plácido de
Castro" vira "Pl?cido de Castro" — e aí o nome do aeroporto chega quebrado no
produto final, que é justamente o que o cliente vai ler.

**`quote = SEM_ASPAS`** (o caractere NUL, `chr(0)`). O arquivo **não usa aspas** para
delimitar campo — mas usa o caractere `"` como símbolo de *segundo* nas coordenadas:
`09°52'06"S`. Com o `quote='"'` padrão, o Spark abre uma aspa ali e sai
engolindo linhas até achar a próxima. Desligar o quoting (apontando para um
caractere que não existe no arquivo) é o que mantém uma linha = um registro.


In [0]:
# Configuração do arquivo de Aeródromos

# Lê o arquivo CSV de aeródromos públicos com configurações específicas
aerodromos = (
    spark.read.format("csv")
    .option("sep", ";")                       # Separador brasileiro (ponto-e-vírgula)
    .option("header", "true")                 # Primeira linha válida contém os nomes das colunas
    .option("skipRows", 1)                    # Pula a primeira linha (cabeçalho informativo)
    .option("encoding", "ISO-8859-1")         # Encoding latin-1, NÃO UTF-8 (mantém acentos corretos)
    .option("quote", SEM_ASPAS)               # Desliga o quoting: aspas aqui são "segundos" nas coordenadas
    .load(f"{REF}/AerodromosPublicos.csv")    # Carrega o arquivo de aeródromos
)

# Seleciona e renomeia as colunas relevantes (normalização para snake_case)
# Colunas entre crases (`) contêm espaços ou caracteres especiais no nome original
aerodromos = aerodromos.select(
    F.col("`Código OACI`").alias("icao"),                # Código ICAO do aeroporto (ex: SBGR)
    F.col("CIAD").alias("ciad"),                          # Código CIAD
    F.col("Nome").alias("nome"),                          # Nome do aeroporto
    F.col("`Município`").alias("municipio"),             # Município onde está localizado
    F.col("UF").alias("uf"),                             # Estado (UF)
    F.col("`Município Servido`").alias("municipio_servido"),  # Município atendido
    F.col("`UF Servido`").alias("uf_servido"),                # Estado atendido
    F.col("Latitude").alias("latitude"),                 # Coordenada de latitude
    F.col("Longitude").alias("longitude"),               # Coordenada de longitude
    F.col("Altitude").alias("altitude"),                 # Altitude do aeródromo
    F.col("`Situação`").alias("situacao"),               # Situação operacional
).withColumn("_ingerido_em", F.current_timestamp())      # Adiciona timestamp de ingestão

# Grava a tabela de aeródromos no formato Delta Lake
aerodromos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"  # Permite alteração no schema se necessário
).saveAsTable("voebem.bronze.aerodromos")

# Exibe o total de linhas carregadas
print(f"bronze.aerodromos: {spark.table('voebem.bronze.aerodromos').count():,} linhas")

# Query de validação: exibe alguns aeroportos conhecidos para conferência
display(spark.sql("SELECT icao, nome, municipio, uf FROM voebem.bronze.aerodromos WHERE icao IN ('SBRB','SBGR','SBSP','SBFZ')"))


## 2. Empresas — DOIS cadastros, DUAS tabelas

A ANAC publica empresas aéreas em dois arquivos distintos, em duas pastas distintas
do portal: um cadastro de empresas **nacionais** e outro de **estrangeiras**. São dois
processos administrativos diferentes, com portarias diferentes.

A tentação aqui é grande: os dois têm **exatamente** o mesmo cabeçalho
(`"ICAO";"Estrangeira";"Razao";"Servico";...`), então um `UNION` sairia de graça.

**E é exatamente por isso que a gente não faz.** No bronze, uma tabela por arquivo de
origem. Se amanhã a ANAC republicar só o cadastro de estrangeiras, eu quero conseguir
reprocessar só ele e comparar com a versão anterior. Um `UNION` no bronze apaga a
fronteira entre as duas fontes e me obriga a reprocessar as duas juntas para sempre.

A unificação **vai** acontecer — é justamente o exemplo clássico de "mesmo assunto,
dois sistemas". Só que ela é decisão de modelagem, e o lugar dela é a silver.

Estes dois são UTF-8 com BOM e **usam** aspas de verdade — ou seja, configuração
oposta à do arquivo anterior, que veio do mesmo portal.

> Nota de campo: existe um `pda_empresas_aereas_nacionais.csv` na **raiz** de
> `Operador Aéreo/` que está corrompido na origem (144 MB, cadastro repetido
> centenas de vezes). O bom está na subpasta `Empresas Aereas Nacionais/`.
> Consulte docs/fontes.md.


In [0]:
# Define função para ler cadastros de empresas aéreas
# Mantém DUAS tabelas separadas (nacionais e estrangeiras) porque são processos administrativos diferentes
def ler_empresas(arquivo: str):
    """Lê um cadastro de empresas. Sem união, sem enriquecimento: uma tabela por arquivo."""
    return (
        spark.read.format("csv")
        .option("sep", ";")              # Separador brasileiro (ponto-e-vírgula)
        .option("header", "true")        # Primeira linha válida contém os nomes das colunas
        .option("skipRows", 1)           # Pula a primeira linha (cabeçalho informativo)
        .option("encoding", "UTF-8")     # Codificação UTF-8
        .option("quote", '"')            # Campos entre aspas duplas
        .load(f"{REF}/{arquivo}")        # Carrega o arquivo especificado
        .select(
            F.col("ICAO").alias("icao"),                    # Código ICAO da empresa (ex: GLO, TAM, AZU)
            F.col("Estrangeira").alias("sigla_iata"),       # Sigla IATA
            F.col("Razao").alias("razao_social"),           # Razão social da empresa
            F.col("Servico").alias("servico"),              # Tipo de serviço prestado
            F.col("Cidade").alias("cidade"),                # Cidade sede
            F.col("UF").alias("uf"),                        # Estado (UF) sede
            F.col("Ativa").alias("situacao"),               # Situação (ATIVA/INATIVA)
        )
        .withColumn("_arquivo_origem", F.lit(arquivo))      # Nome do arquivo de origem
        .withColumn("_ingerido_em", F.current_timestamp())  # Timestamp de ingestão
    )

# Loop que processa os dois arquivos de empresas (nacionais e estrangeiras)
# Cada um gera uma tabela bronze separada
for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv",    "voebem.bronze.empresas_nacionais"),
    ("pda_empresas_aereas_estrangeiros.csv", "voebem.bronze.empresas_estrangeiras"),
]:
    # Lê o arquivo, grava no formato Delta e exibe o total de linhas
    ler_empresas(arquivo).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"  # Permite alteração no schema se necessário
    ).saveAsTable(tabela)
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
# Query de validação: compara as duas tabelas de empresas
# Mostra total de linhas e quantas têm código ICAO preenchido
# (empresas estrangeiras geralmente têm ICAO; nacionais nem todas)
display(spark.sql("""
    SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voebem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras', COUNT(*),
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM voebem.bronze.empresas_estrangeiras
"""))

In [0]:
# Query de validação: exibe algumas empresas nacionais conhecidas (GOL, TAM, Azul, Passaredo)
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_nacionais
    WHERE icao IN ('GLO','TAM','AZU','PAM')
    ORDER BY icao
"""))

# Query de validação: exibe algumas empresas estrangeiras conhecidas (American, TAP, Avianca, Aerolíneas Argentinas)
display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM voebem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL','TAP','AVA','ARG')
    ORDER BY icao
"""))


## 3. Códigos de operação — seed table

O VRA guarda `codigo_di = "0"` e `codigo_tipo_linha = "N"`. Sem tradução, isso
não significa nada para um analista — e significa menos ainda para um LLM.

A ANAC publica essas descrições numa **página HTML**, não num CSV. Então esta
tabela é uma *seed table*: dado de referência pequeno, estável e curado à mão,
versionado junto com o código. É uma categoria legítima de fonte — o erro seria
deixar esse mapeamento espalhado em `CASE WHEN` dentro das queries.


In [0]:
# Define uma "seed table" (tabela semente) com os códigos de operação
# A ANAC publica estas descrições em HTML, não em CSV, então criamos manualmente
# Esta tabela traduz códigos do VRA em descrições legíveis
CODIGOS = [
    # Códigos DI (Autorização) - indicam o tipo de etapa do voo
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    # Códigos de Tipo de Linha - indicam se é doméstico/internacional e passageiro/carga
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

# Cria o DataFrame a partir da lista de tuplas, com schema explícito
codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")

# Grava a tabela no formato Delta Lake
codigos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"  # Permite alteração no schema se necessário
).saveAsTable("voebem.bronze.codigos_operacao")

# Exibe o total de linhas e o conteúdo completo da tabela para validação
print(f"bronze.codigos_operacao: {spark.table('voebem.bronze.codigos_operacao').count()} linhas")
display(spark.table("voebem.bronze.codigos_operacao"))

## 4. O bronze fechado: 4 arquivos de referência, 4 tabelas, nenhuma união

In [0]:
display(spark.sql("SHOW TABLES IN voebem.bronze"))

## 5. O que o Delta guardou sem a gente pedir

A gente nunca escreveu uma linha de código de versionamento. Mesmo assim:

In [0]:
display(spark.sql("""
    SELECT version, timestamp, operation,
           operationMetrics.numOutputRows AS linhas_escritas
    FROM (DESCRIBE HISTORY voebem.bronze.vra)
    ORDER BY version
"""))

## 6. Time travel

A versão 0 é a primeira carga do marco-03; a carga seguinte é a segunda execução
da ingestão (aquela que provou a idempotência). Dá para consultar as duas lado a lado.

In [0]:
display(spark.sql("""
    SELECT 'versao 0 (1a carga)'   AS versao,
           COUNT(*)                AS linhas,
           MIN(ingerido_em)       AS ingerido_em
    FROM voebem.bronze.vra VERSION AS OF 0
    UNION ALL
    SELECT 'versao atual', COUNT(*), MIN(ingerido_em)
    FROM voebem.bronze.vra
"""))

Mesmo número de linhas, `ingerido_em` diferente: a prova de idempotência do
marco anterior, agora reconstruída **do histórico**, sem ter guardado nada.

Isso é propriedade do formato de tabela aberto, não código nosso. Todo `write`
no Delta grava um commit no log de transações; o dado antigo continua nos
arquivos Parquet até um `VACUUM`. Auditoria e rollback saem de graça.


In [0]:
for tabela, comentario in [
    ("voebem.bronze.aerodromos",
     "Bronze - cadastro de aerodromos publicos da ANAC, como chegou. Chave: codigo ICAO (OACI). "
     "Cobre apenas aerodromos brasileiros - aeroportos estrangeiros do VRA nao estao aqui."),
    ("voebem.bronze.empresas_nacionais",
     "Bronze - cadastro de empresas aereas NACIONAIS da ANAC, como chegou. Chave: codigo ICAO. "
     "Nao unir com empresas_estrangeiras nesta camada: a uniao e feita na silver."),
    ("voebem.bronze.empresas_estrangeiras",
     "Bronze - cadastro de empresas aereas ESTRANGEIRAS autorizadas a operar no Brasil, como chegou. "
     "Chave: codigo ICAO. Cadastro separado do nacional na origem, mantido separado no bronze."),
    ("voebem.bronze.codigos_operacao",
     "Bronze - seed table curada a partir da pagina de descricao de variaveis da ANAC. "
     "Traduz codigo_di e codigo_tipo_linha para descricao em portugues."),
]:
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")

print("comentarios aplicados")

In [0]:
# 📝 GUIA: Como exportar notebooks como .py para o Git

print("""
═══════════════════════════════════════════════════════════════════════════════
🔧 OPÇÃO 1: Usar Databricks CLI (RECOMENDADO)
═══════════════════════════════════════════════════════════════════════════════

No seu terminal local (onde está o repositório Git clonado):

# 1. Instale o Databricks CLI (se ainda não tiver)
pip install databricks-cli

# 2. Configure a autenticação (uma vez só)
databricks configure --token
   Host: https://<seu-workspace>.cloud.databricks.com
   Token: <seu-personal-access-token>

# 3. Exporte os notebooks como .py
databricks workspace export \\
  /Users/lohana472@gmail.com/imersao-engenharia-de-dados/voebem/notebooks/03_bronze_vra \\
  ./03_bronze_vra.py \\
  --format SOURCE

databricks workspace export \\
  /Users/lohana472@gmail.com/imersao-engenharia-de-dados/voebem/notebooks/04_bronze_referencias \\
  ./04_bronze_referencias.py \\
  --format SOURCE

# 4. Faça o commit dos arquivos .py
git add *.py
git commit -m "feat: exporta notebooks bronze como .py"
git push

═══════════════════════════════════════════════════════════════════════════════
🔄 OPÇÃO 2: Script Python com Databricks SDK
═══════════════════════════════════════════════════════════════════════════════

Crie um arquivo export_notebooks.py localmente:
"""
)

script = '''
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat
import os

# Inicializa o cliente (usa variáveis de ambiente DATABRICKS_HOST e DATABRICKS_TOKEN)
w = WorkspaceClient()

notebooks = [
    "/Users/lohana472@gmail.com/imersao-engenharia-de-dados/voebem/notebooks/03_bronze_vra",
    "/Users/lohana472@gmail.com/imersao-engenharia-de-dados/voebem/notebooks/04_bronze_referencias"
]

for notebook_path in notebooks:
    notebook_name = notebook_path.split("/")[-1]
    output_file = f"{notebook_name}.py"
    
    # Exporta como SOURCE (Python puro)
    content = w.workspace.export(notebook_path, format=ExportFormat.SOURCE)
    
    # Salva localmente
    with open(output_file, "wb") as f:
        f.write(content.content)
    
    print(f"✅ Exportado: {output_file}")

print("\n🎉 Pronto! Agora faça: git add *.py && git commit && git push")
'''

print(script)

print("""
═══════════════════════════════════════════════════════════════════════════════
⚙️ OPÇÃO 3: Configurar Git Folder para sincronizar como .py automaticamente
═══════════════════════════════════════════════════════════════════════════════

1. Vá em Workspace → Repos → (seu repositório)
2. Clique nas configurações (⚙️) do Repo
3. Em "Notebook Format", escolha "SOURCE" em vez de "DBC" ou "JUPYTER"
4. Agora todos os notebooks serão sincronizados como .py automaticamente

═══════════════════════════════════════════════════════════════════════════════
📊 COMPARAÇÃO DOS FORMATOS
═══════════════════════════════════════════════════════════════════════════════

Formato .ipynb (Jupyter):                Formato .py (SOURCE):
✓ Preserva outputs das células         ✓ Código mais limpo
✓ Melhor para colaboração visual       ✓ Diff no Git muito mais legível
✓ Mantém markdown formatado             ✓ Arquivos menores
✗ Arquivos grandes (com outputs)        ✓ Melhor para code review
✗ Diff no Git poluído com metadata      ✓ Compatível com ferramentas Python
                                         ✗ Perde outputs das células
                                         ✗ Markdown vira comentários #

═══════════════════════════════════════════════════════════════════════════════
💡 RECOMENDAÇÃO
═══════════════════════════════════════════════════════════════════════════════

Para código de produção (pipelines de dados):
  → Use formato .py (OPÇÃO 1 ou 3)
  → Mais fácil de versionar e revisar no Git
  → Melhor para CI/CD

Para análises exploratórias e documentação:
  → .ipynb pode ser melhor
  → Outputs ajudam na compreensão

""")